In [2]:
import pandas as pd
from sts.mktdata.ticker import Ticker
import numpy as np

In [12]:
asset = 'usetf'

In [19]:
df = pd.read_excel(f'/home/yuqing42/Dropbox/stsinvest/research/trend/trend_top_signal/{asset}s_20260227.xlsx', sheet_name = 'Sheet1')
df = df.rename(columns = {'Unnamed: 0':'date'})
df['volMul'] = 50 / (df['vol'] * np.sqrt(252) * 100)
df = df[( df['volMul'] < 2 ) & (df['volMul'] > 0.5) & (df['TSScore'].abs() > 0.3)].copy()  #TODO filters

ticker = Ticker('us/etf.yml') if asset == 'usetf' else Ticker('us/stock.yml')
top_symbols = ticker.top_trading_symbols('20260227', top_n = 1000)
symbol_me_df = ticker.history(top_symbols, '20260227', '20260227', interval = '1ME')
symbol_me_df
symbol_me_df['trdVal'] = symbol_me_df['volume'] * symbol_me_df['close']
df2 = df.merge(symbol_me_df[['symbol', 'volume', 'trdVal']], on = 'symbol', how = 'left')
df2.sort_values('trdVal', ascending=False, inplace = True)
corr_df = pd.read_excel(f'/home/yuqing42/Dropbox/stsinvest/research/trend/trend_top_signal/trend_top_signal_correlation_{asset}s_2026-02-27 00:00:00.xlsx', sheet_name = 'Sheet1')
corr_df = corr_df.set_index('symbol')
out = {}
for n, row in df2.iterrows():
    symbol = row['symbol']
    flag = False
    for key in out:
        cor = corr_df.loc[symbol, key]
        if abs(cor) > 0.7:
            flag = True
            break
    if not flag:
        out[symbol] = row
filtered_df = pd.DataFrame(out.values())
cols = ['symbol', 'TSScore', 'volMul', 'TS0', 'TrendTop20']
cols2 = cols + [c for c in filtered_df.columns if c not in cols]

In [20]:
t = filtered_df[cols2].dropna().sort_values('TSScore', ascending=False)
t.to_csv(f'~/Dropbox/stsinvest/research/trend/trend_top_signal/filtered_trend_signal_{asset}.csv', index = False)

In [21]:
t

,symbol,TSScore,volMul,TS0,TrendTop20,date,TS1,TS2,abVolume,abPrice,TrendTop,TrendTop10,vol,volume,trdVal
77,EWY,1.584202,1.392267,1.352943,1.534508,2026-02-27,1.847683,1.485634,0.426214,0.125236,0,1.534508,0.022623,65090069.0,7.824477e+09
83,GLD,1.013652,1.619815,0.427587,4.363866,2026-02-27,1.004267,1.215265,0.048421,-0.209249,0,0.000000,0.019445,110005019.0,4.860902e+10
108,COPX,0.970957,0.992159,0.538612,0.000000,2026-02-27,1.045678,1.065259,0.129440,-0.097986,0,0.000000,0.031746,33111949.0,2.692332e+09
103,REMX,0.820009,0.896399,0.561722,0.000000,2026-02-27,0.752296,0.951246,-0.010092,-0.122976,0,0.000000,0.035137,7624766.0,6.226384e+08
34,EWZ,0.797645,1.614435,0.476701,0.000000,2026-02-27,1.058506,0.730719,-0.081640,-0.109001,0,0.000000,0.019510,195456604.0,7.214303e+09
35,ECH,0.761175,1.646848,-0.138066,0.000000,2026-02-27,0.822013,1.020363,0.416261,0.049667,0,0.000000,0.019126,6160409.0,2.783889e+08
20,EWW,0.733336,1.655149,0.360765,0.000000,2026-02-27,0.916668,0.735304,0.771783,0.073648,0,0.000000,0.019030,8093283.0,6.218069e+08
86,PPLT,0.710371,0.845602,0.231276,0.000000,2026-02-27,0.495094,1.013587,-0.181287,-0.305581,0,0.000000,0.037248,4400505.0,7.998798e+08
63,ITA,0.704979,1.696048,0.405558,0.000000,2026-02-27,0.710052,0.801403,0.066359,-0.116282,0,0.000000,0.018571,4701796.0,1.063452e+09
72,XLE,0.672236,1.644151,0.683889,0.000000,2026-02-27,1.076305,0.398973,0.249436,0.135986,0,0.000000,0.019157,275455100.0,1.438151e+10


'usetf'